# 02 — Skill Sweep

Sweep Tactics from 50 to 120 and visualise how mean damage scales.

**Key insight:** Warrior damage formula uses `(Anatomy + Tactics) / 2`,
NOT the weapon skill (Swordsmanship). Sweeping Swordsmanship will
affect hit chance but not the damage multiplier.

In [ ]:
from pathlib import Path

from omega.model.constants import SKILLID_SWORDSMANSHIP, SKILLID_TACTICS, SKILLID_ANATOMY
from omega.shard import ShardData
from omega.simulation import (
    ArmorSpec, CombatantSpec, ParameterSweep, Scenario, Variable, WeaponSpec, run_sweep,
)
from omega.reporting.tables import summary_table, format_table_html
from omega.reporting.plots import damage_vs_parameter

# Works whether CWD is the project root or the notebooks/ directory
SHARD_ROOT = Path("submodules/zuluhotel_omega_2.5")
if not SHARD_ROOT.exists():
    SHARD_ROOT = Path("../submodules/zuluhotel_omega_2.5")
shard = ShardData.from_path(SHARD_ROOT)
parse_results = shard.parse_combat_scripts()

In [ ]:
sweep = ParameterSweep(
    scenario=Scenario(
        attacker=CombatantSpec(
            name="Warrior",
            skills={SKILLID_SWORDSMANSHIP: 100, SKILLID_TACTICS: 100, SKILLID_ANATOMY: 100},
            str_=100, dex_=100, int_=25,
            class_levels={"IsWarrior": 5},
            weapon=WeaponSpec(name="Broadsword", damage="3d6+2"),
        ),
        defender=CombatantSpec(
            name="Target",
            is_npc=True,
            str_=50, dex_=50, int_=50,
            hp=500,
            armor=ArmorSpec(ar=30),
        ),
        iterations=100,
        base_seed=1234,
    ),
    variables=(
        Variable.from_range("attacker", f"skills.{SKILLID_TACTICS}", start=50, stop=120, step=10),
    ),
)

result = run_sweep(
    sweep,
    parse_results=parse_results,
    config_resolver=shard.resolve_config_path,
    em_modules_dir=shard.root / "scripts" / "modules",
)
print(f"{len(result.cells)} cells completed in {result.total_time:.1f}s")

In [ ]:
# Damage vs Tactics curve with p5-p95 shading
damage_vs_parameter(result, f"attacker.skills.{SKILLID_TACTICS}", title="Damage vs Tactics")

In [ ]:
# Full summary table
from IPython.display import HTML

rows = summary_table(result, stats=["mean", "median", "min", "max", "p5", "p95", "total", "hit_rate"])
HTML(format_table_html(rows))